In [1]:
import matplotlib.pyplot as plt 
import pandas as pd 
import seaborn as sns 
import numpy as np
import yaml
import pickle
from itertools import product, cycle
import os, sys
from tqdm.notebook import tqdm
from scipy import stats
import scikit_posthocs as sp

from pathlib import Path
sys.path.insert(0, str(Path().resolve().parents[1]))

*Analysis to perform*

1. Summary table of mean +- std of performances Dataset/Method/Metric
2. Analysis binned by total missing ratio;
3. Analysis binned by MNAR percentage; 

In [2]:
# load simulation results

# !!!! Need to change output fiel to current timestamp

simulation_path = "../output/ablation_results06_03_2026.pkl"

with open(simulation_path, 'rb') as input_file:
    simulation_results = pickle.load(input_file)

In [3]:
# Load simulation configuration

with open("../config/simulation_config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# conf is a tuple (dataset_id,props,mf_proportions,mnar_proportions, seed)

configs = list(product(
    cfg['dataset_ids'],
    cfg['md_param_grid']['props'],
    cfg['md_param_grid']['mf_proportions'],
    cfg['md_param_grid']['mnar_proportions'],
    range(cfg['n_runs'])
    ))

In [4]:
TESTED_METHODS = [
        'mige_no_proj',
        'mige_proj',
    ]

TESTED_METHOD_STATS = [
    'mige_no_proj',
    'mige_proj',
]

In [5]:
# configs of dataset = dataset_id
# dataset_id = configs[0][0]
# key_subset = {k: v for k, v in d.items() if k[0] == X} this is for dictionaries
# conf_subset = [conf for conf in configs if conf[0] == dataset_id]

cfg['dataset_ids']

[33, 45, 17, 15, 174, 544]

In [6]:
def read_pickle(path):
    with open(path, 'rb') as input_file:
        return pickle.load(input_file)

In [8]:
# simulation results is a dictionary structured as such:
# simulation_results[conf][metric_type] metric_type in {external_metrics, internal_metrics}
# conf is a tuple (dataset_id,props,mf_proportions,mnar_proportions, seed)
# subset dictionary for first element subset = {k: v for k, v in d.items() if k[0] == X}

In [8]:
# tested external metrics 
external_metrics = list(simulation_results[configs[0]]['external_metrics']['mige_no_proj'])
# tested internal metrics 
internal_metrics = list(simulation_results[configs[0]]['internal_metrics']['mige_no_proj'])
# tested methods
methods = list(simulation_results[configs[0]]['internal_metrics'])

In [7]:
def safe_get(v, key1, key2):
    """
    This function is a safe get with double key because some are not list but single np.nan
    """
    try:
        return v[key1][key2]
    except:
        return np.nan

def spaghetti_dataframe_(df, metric_name):
    """
    This function is specific to pivot a dataframe into long format
    """
    df_long = (
        df.T
        .assign(metric=metric_name)
        .set_index("metric", append=True)
        .stack()
    )
    return df_long

In [9]:
def combine_dataset(results_df, metrics):
    """
    From aggregated result datasets with the form: 
    {metric: {dataset: {method: <value>}}}
    """
    long_dfs = {
    metric: spaghetti_dataframe_(
            df = pd.DataFrame.from_dict(results_df[metric]),
            metric_name = metric
        )
    for metric in metrics
    }
    combined = pd.DataFrame(pd.concat([df for df in long_dfs.values()]))
    combined.index = combined.index.set_names(['dataset','metric','method'])
    combined = combined.rename({0: "value"}, axis = 'columns') 
    combined = combined.unstack(level = -1)
    return combined

# External metrics aggregates

In [10]:
# build dictionaries to store:
# - results (lists for each of the 10 runs)
# - means and stds (single value)
# dictionary form {metric: {dataset: {method: <value>}}}

external_metrics_results = {met:{d:{m:[] for m in methods} for d in cfg['dataset_ids']} for met in external_metrics}
external_metrics_means = {met:{d:{m:np.nan for m in methods} for d in cfg['dataset_ids']} for met in external_metrics}
external_metrics_stds = {met:{d:{m:np.nan for m in methods} for d in cfg['dataset_ids']} for met in external_metrics}

for k1,v1 in simulation_results.items():
    dataset_id = k1[0]
    # collect results
    for metric in external_metrics:
        for method in methods:
            #res = v1['external_metrics'][method][metric]
            res = safe_get(v1['external_metrics'],method,metric)
            external_metrics_results[metric][dataset_id][method].append(res)

# calculate means and standard deviations
for metric,v1 in external_metrics_results.items():
    for dataset_id, v2 in v1.items():
        for method,v3 in v2.items():
            external_metrics_means[metric][dataset_id][method] = np.mean(v3)
            external_metrics_stds[metric][dataset_id][method] = np.std(v3)

In [11]:
combined_external_means = combine_dataset(external_metrics_means,external_metrics)
combined_external_stds = combine_dataset(external_metrics_stds,external_metrics)
combined_external_means.head(5)

value          
method         mige_no_proj mige_proj
dataset metric                       
15      ami        0.758106  0.758475
        ari        0.854217  0.853971
        cs         0.758572  0.758747
        vm         0.758380  0.758750
17      ami        0.691453  0.694677

## Ranks for external metrics

In [12]:
combined_external_means['value'][TESTED_METHODS].rank(axis = 'columns', ascending = False)

method          mige_no_proj  mige_proj
dataset metric                         
15      ami              2.0        1.0
        ari              1.0        2.0
        cs               2.0        1.0
        vm               2.0        1.0
17      ami              2.0        1.0
        ari              1.0        2.0
        cs               2.0        1.0
        vm               2.0        1.0
33      ami              2.0        1.0
        ari              2.0        1.0
        cs               2.0        1.0
        vm               2.0        1.0
45      ami              1.0        2.0
        ari              1.0        2.0
        cs               1.0        2.0
        vm               1.0        2.0
174     ami              2.0        1.0
        ari              1.0        2.0
        cs               2.0        1.0
        vm               2.0        1.0
544     ami              1.0        2.0
        ari              1.0        2.0
        cs               1.0        2.0
        vm               1.0        2.0

In [13]:
# Calculate average rank for each method
combined_external_means['value'][TESTED_METHODS].rank(axis = 'columns', ascending = False).mean(axis=0)

method
mige_no_proj    1.541667
mige_proj       1.458333
dtype: float64

In [15]:
combined_external_means['value'][TESTED_METHODS].to_csv("../output/analysis/ablation_external_means.csv")
combined_external_stds['value'][TESTED_METHODS].to_csv("../output/analysis/ablation_external_stds.csv")
combined_external_means['value'][TESTED_METHODS].rank(axis = 'columns', ascending = False).to_csv("../output/analysis/combined_external_ranks.csv")

# Statistical tests

We run friedman test per dataset 

- each dataset provides a Friedman test, as it has 120 runs

In [14]:
dataset_ids = [k for k in external_metrics_results['ari'].keys()]
dataset_ids

[33, 45, 17, 15, 174, 544]

In [15]:
# for ablation study i just compare with wilcoxon for each dataset and adjust with bonferroni

TESTED_METHOD_STATS

['mige_no_proj', 'mige_proj']

In [16]:
statistical_test_results = {}
statistical_test_results['ari'] = {}
statistical_test_results['ami'] = {}

In [17]:

for metric, dataset in product(['ari','ami'],dataset_ids):

    x = external_metrics_results[metric][dataset]['mige_proj']
    y = external_metrics_results[metric][dataset]['mige_no_proj']

    stat, pval = stats.ttest_rel(x,y,alternative='greater')

    print(f"{metric} dataset {dataset}: {pval}")


ari dataset 33: 0.048519142220239085
ari dataset 45: 0.9931243854870898
ari dataset 17: 0.9999920919397299
ari dataset 15: 0.619923179597137
ari dataset 174: 0.7349626251790804
ari dataset 544: 0.9999985148929041
ami dataset 33: 0.0472580360507264
ami dataset 45: 0.9999982832707891
ami dataset 17: 0.008346165676859967
ami dataset 15: 0.36967473231519987
ami dataset 174: 0.04150744618847974
ami dataset 544: 0.999258432491


In [25]:
for metric in ['ari','ami']:
    for dataset in dataset_ids:

        temp = external_metrics_results[metric][dataset]

        test_values = []
        for group in TESTED_METHOD_STATS:
            test_values.append(temp[group])

        stat, p = stats.friedmanchisquare(*test_values)
        posthoc_p = sp.posthoc_nemenyi_friedman(pd.DataFrame(test_values).T)
        posthoc_p.columns = TESTED_METHOD_STATS
        posthoc_p.index = TESTED_METHOD_STATS


        statistical_test_results[metric][dataset] = p
        statistical_test_results_posthoc[metric][dataset] = posthoc_p

